# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
# 1a. Thống kê split + kiểm tra rò rỉ phiên quay
import os, sys, json, statistics
from collections import Counter
from pathlib import Path
sys.path.insert(0, os.getcwd())
from scripts.labcommon import CLASSES, read_jsonl

DATA = Path('data')
SPLITS = ('train', 'val', 'test')
rows = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in SPLITS}
meta = json.loads((DATA / 'splits' / 'meta.json').read_text(encoding='utf-8'))
print('dataset_version:', meta['dataset_version'])
print('mẫu 1 dòng train:', json.dumps(rows['train'][0], ensure_ascii=False)[:300], '\n')

# Số ảnh / phiên / box từng lớp mỗi split
print(f"{'split':6s}{'ảnh':>6s}{'phiên':>7s}{'box':>6s}" + ''.join(f'{c:>9s}' for c in CLASSES) + f"{'ảnh rỗng':>10s}")
for s in SPLITS:
    cnt = Counter(o['label'] for r in rows[s] for o in r['objects'])
    n_sess = len({r['session'] for r in rows[s]})
    empty = sum(not r['objects'] for r in rows[s])
    print(f"{s:6s}{len(rows[s]):6d}{n_sess:7d}{sum(cnt.values()):6d}"
          + ''.join(f'{cnt[c]:9d}' for c in CLASSES) + f'{empty:10d}')
    assert all(cnt[c] == meta['counts'][s]['boxes'][c] for c in CLASSES), s   # khớp meta.json

# Không phiên nào nằm ở hai split, không ảnh nào bị lặp
sess = {s: {r['session'] for r in rows[s]} for s in SPLITS}
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    common = sess[a] & sess[b]
    print(f'phiên chung {a:5s}–{b:5s}: {len(common)}')
    assert not common, common
ids = [r['image_id'] for s in SPLITS for r in rows[s]]
assert len(ids) == len(set(ids)), 'ảnh bị lặp giữa các split'

# Kích thước ảnh và kích thước box (diện tích tương đối so với khung hình)
print('\nkích thước ảnh:', dict(Counter((r['width'], r['height']) for s in SPLITS for r in rows[s])))
all_areas = []
for c in CLASSES:
    areas = [o['yolo'][2] * o['yolo'][3] for s in SPLITS for r in rows[s] for o in r['objects'] if o['label'] == c]
    all_areas += areas
    print(f'{c:8s} n={len(areas):5d}  diện tích trung vị = {100 * statistics.median(areas):.2f}% khung hình')
print(f'{"tất cả":8s} n={len(all_areas):5d}  diện tích trung vị = {100 * statistics.median(all_areas):.2f}% khung hình')

In [ ]:
# 1b. Vẽ 6 ảnh (2 ảnh/split) kèm box gốc từ nhãn YOLO → pixel
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

COLORS = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}

def draw_gt(r):
    im = Image.open(DATA / r['image_path']).convert('RGB')
    d = ImageDraw.Draw(im)
    W, H = r['width'], r['height']
    for o in r['objects']:
        xc, yc, w, h = o['yolo']                       # chuẩn hoá [0,1], tâm + kích thước
        box = [(xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H]
        d.rectangle(box, outline=COLORS[o['label']], width=3)
        d.text((box[0] + 3, box[1] + 2), o['label'], fill=COLORS[o['label']])
    return im

rng = random.Random(0)
picks = []
for s in SPLITS:
    multi = [r for r in rows[s] if len({o['label'] for o in r['objects']}) >= 2]   # ưu tiên ảnh có ≥ 2 lớp
    pool = multi if len(multi) >= 2 else rows[s]
    picks += [(s, r) for r in rng.sample(pool, 2)]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
for ax, (s, r) in zip(axes.flat, picks):
    ax.imshow(draw_gt(r)); ax.axis('off')
    ax.set_title(f"[{s}] {r['image_id']} · {r['session']} · {len(r['objects'])} box", fontsize=9)
fig.legend(handles=[plt.Line2D([], [], color=c, lw=3, label=l) for l, c in COLORS.items()],
           loc='lower center', ncol=3)
plt.tight_layout()
Path('report/figures').mkdir(parents=True, exist_ok=True)
fig.savefig('report/figures/cp1_gt_samples.jpg', dpi=80, bbox_inches='tight')
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)
# 2a. Nhãn YOLO → JSON thang 0–1000 + prompt
import json

def yolo_to_box(label, yolo, coord='norm1000', width=640, height=360):
    """YOLO (xc, yc, w, h chuẩn hoá [0,1]) → {"label", "bbox_2d": [x1, y1, x2, y2]} số nguyên.
    coord='norm1000': thang 0–1000 (quy ước grounding của Qwen3-VL); coord='pixel': pixel ảnh (chỉ dùng cho E4)."""
    xc, yc, w, h = yolo
    sx, sy = (1000, 1000) if coord == 'norm1000' else (width, height)
    x1, y1 = (xc - w / 2) * sx, (yc - h / 2) * sy
    x2, y2 = (xc + w / 2) * sx, (yc + h / 2) * sy
    x1, x2 = (min(max(round(v), 0), sx) for v in (x1, x2))
    y1, y2 = (min(max(round(v), 0), sy) for v in (y1, y2))
    # box siêu nhỏ có thể bị làm tròn thành x2 == x1 → score.py coi CẢ câu trả lời là hỏng; nới ra 1 đơn vị
    if x2 <= x1: x1, x2 = (x1, x1 + 1) if x1 < sx else (x1 - 1, x1)
    if y2 <= y1: y1, y2 = (y1, y1 + 1) if y1 < sy else (y1 - 1, y1)
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects, coord='norm1000', width=640, height=360):
    boxes = [yolo_to_box(o['label'], o['yolo'], coord, width, height) for o in objects]
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))   # trên→dưới, trái→phải: thứ tự ổn định cho model học
    return json.dumps(boxes, separators=(',', ':'))                 # "[]" nếu không có object

def make_prompt(coord='norm1000', width=640, height=360):
    if coord == 'norm1000':
        scale = ('integers on a 0-1000 scale relative to the image width and height: '
                 '(0,0) is the top-left corner and (1000,1000) is the bottom-right corner, NOT pixels')
    else:
        scale = f'integer pixel coordinates of this {width}x{height} image: (0,0) is the top-left corner'
    return ('Detect all road damage in this dashcam image. Classes: "pothole", "crack", "manhole".\n'
            'Return ONLY a JSON list, with no explanation and no markdown, in the form '
            '[{"label": "pothole", "bbox_2d": [x1, y1, x2, y2]}].\n'
            f'Coordinates are {scale}.\n'
            'If there is no damage, return [].')

PROMPT = make_prompt('norm1000')
print(PROMPT, '\n')
from scripts.labcommon import read_jsonl
_r = read_jsonl('data/splits/train.jsonl')[0]
print('nhãn YOLO :', _r['objects'])
print('đáp án    :', objects_to_answer(_r['objects']))

In [ ]:
# 2b. Kiểm tra: JSON → pixel phải trùng box gốc; trọng tài đọc được nhãn; lệch thang làm F1 sụp
import os, sys
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl
from scripts.score import parse_output, evaluate

DATA = Path('data')
rows = {s: read_jsonl(DATA / 'splits' / f'{s}.jsonl') for s in ('train', 'val', 'test')}

def gt_pixel(r):      # box gốc YOLO → pixel (kẹp trong ảnh)
    W, H = r['width'], r['height']
    return [(max((xc - w / 2) * W, 0), max((yc - h / 2) * H, 0), min((xc + w / 2) * W, W), min((yc + h / 2) * H, H))
            for xc, yc, w, h in (o['yolo'] for o in r['objects'])]

def answer_to_pixel(answer, W, H):   # JSON 0–1000 → pixel, đọc bằng chính parse_output của trọng tài
    return [(x1 / 1000 * W, y1 / 1000 * H, x2 / 1000 * W, y2 / 1000 * H) for x1, y1, x2, y2 in
            (p['bbox'] for p in parse_output(answer))]

# (1) Round-trip trên TẤT CẢ ảnh: mỗi box gốc phải có box quy ngược lệch < 1 pixel
max_err, n_box = 0.0, 0
for rs in rows.values():
    for r in rs:
        ans = objects_to_answer(r['objects'])
        assert parse_output(ans) is not None, (r['image_id'], ans)       # trọng tài đọc được
        back = answer_to_pixel(ans, r['width'], r['height'])
        assert len(back) == len(r['objects'])
        for g in gt_pixel(r):
            err = min(max(abs(a - b) for a, b in zip(g, p)) for p in back)
            max_err, n_box = max(max_err, err), n_box + 1
print(f'round-trip {n_box} box: sai số lớn nhất = {max_err:.2f} px (do làm tròn về số nguyên 0–1000)')
assert max_err < 1.0

# (2) Vẽ lại: box gốc (xanh lá, dày) và box quy ngược từ JSON (đỏ, mảnh) phải chồng khít
r = max(rows['test'], key=lambda r: len(r['objects']))
im = Image.open(DATA / r['image_path']).convert('RGB'); d = ImageDraw.Draw(im)
for g in gt_pixel(r):
    d.rectangle(g, outline='lime', width=5)
for p in answer_to_pixel(objects_to_answer(r['objects']), r['width'], r['height']):
    d.rectangle(p, outline='red', width=1)
plt.figure(figsize=(12, 7)); plt.imshow(im); plt.axis('off')
plt.title(f"{r['image_id']}: xanh = box gốc YOLO, đỏ = JSON 0–1000 quy ngược về pixel\n"
          f"{objects_to_answer(r['objects'])[:150]}…", fontsize=9)
Path('report/figures').mkdir(parents=True, exist_ok=True)
plt.savefig('report/figures/cp2_roundtrip.jpg', dpi=80, bbox_inches='tight'); plt.show()

# (3) Chấm "đáp án hoàn hảo" bằng trọng tài: đúng thang → F1 = 1; lệch thang → F1 sụp (bằng chứng Câu 3)
print('\nchấm đáp án hoàn hảo trên test bằng scripts/score.py:')
for label_coord, score_coord in [('norm1000', 'norm1000'), ('pixel', 'pixel'),
                                 ('norm1000', 'pixel'), ('pixel', 'norm1000')]:
    preds = {r['image_id']: objects_to_answer(r['objects'], label_coord, r['width'], r['height'])
             for r in rows['test']}
    m = evaluate(rows['test'], preds, coord=score_coord, n_boot=200)
    print(f"  nhãn {label_coord:8s} | chấm --coord {score_coord:8s} → F1 = {m['micro']['f1']:.3f}  "
          f"mean IoU = {m['mean_iou']:.3f}  FP/ảnh = {m['fp_per_image']:.2f}")

# (4) Độ dài đáp án theo token: đáp án dài hơn max_new_tokens (256) sẽ bị cắt → JSON hỏng
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-VL-4B-Instruct')
for s, rs in rows.items():
    lens = [len(tok(objects_to_answer(r['objects'])).input_ids) + 1 for r in rs]   # +1 cho <|im_end|>
    nbox = [len(r['objects']) for r in rs]
    print(f'{s:5s} token đáp án: trung bình {sum(lens) / len(lens):5.1f}, lớn nhất {max(lens):4d} '
          f'(ảnh {max(nbox)} box) | số ảnh > 256 token: {sum(l > 256 for l in lens)}')
print(f'≈ {(lens[nbox.index(max(nbox))] - 2) / max(max(nbox), 1):.0f} token mỗi box')


## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)
# 3a. Nạp Qwen3-VL-4B 4-bit (ViT + lm_head giữ fp16) và suy luận zero-shot trên test
import os, sys, json, time, shutil
from collections import Counter
from pathlib import Path
import torch
from PIL import Image
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
sys.path.insert(0, os.getcwd())
from scripts.labcommon import read_jsonl

assert 'PROMPT' in globals(), 'Chạy ô 2a trước (định nghĩa PROMPT, objects_to_answer)'
MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
DATA = Path('data')
DRIVE_DIR = Path('/content/drive/MyDrive/lab10_2A202602818')   # sao lưu: session Colab có thể bị ngắt

def load_base_model():
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=['visual', 'lm_head'])   # ViT (cả projector) + lm_head giữ fp16
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={'': 0})
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.tokenizer.padding_side = 'left'                               # batch generate: pad bên trái
    return model, processor

if 'model' not in globals():          # chạy lại ô thì không nạp lại model (≈ 2–3 phút)
    t0 = time.time()
    model, processor = load_base_model()
    print(f'nạp model: {time.time() - t0:.0f}s | VRAM {torch.cuda.memory_allocated() / 1e9:.2f} GB')
model.eval()

# Xác nhận: Linear của ViT/projector và lm_head là fp16 thường, của LLM là 4-bit
kinds = Counter()
for n, m in model.named_modules():
    if isinstance(m, torch.nn.Linear):
        part = 'visual' if '.visual.' in f'.{n}.' else 'lm_head' if n.endswith('lm_head') else 'LLM'
        kinds[(part, type(m).__name__, str(m.weight.dtype))] += 1
for k, v in sorted(kinds.items()):
    print(f'  {k[0]:8s} {k[1]:10s} {k[2]:15s} × {v}')
assert all(t == 'Linear' for (part, t, _) in kinds if part == 'visual'), 'ViT bị lượng tử hoá!'

def generate_batch(images, prompt, max_new_tokens=256):
    msgs = [[{'role': 'user', 'content': [{'type': 'image', 'image': im},
                                          {'type': 'text', 'text': prompt}]}] for im in images]
    inputs = processor.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_dict=True,
                                           return_tensors='pt', padding=True).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def predict(rows, out_path, prompt, batch_size=4, max_new_tokens=256):
    """Ghi {"image_id", "raw"} từng dòng. Chạy lại thì bỏ qua ảnh đã có (khôi phục từ Drive nếu mất work/)."""
    out_path = Path(out_path); out_path.parent.mkdir(parents=True, exist_ok=True)
    backup = DRIVE_DIR / out_path.name
    if not out_path.exists() and backup.exists():
        shutil.copy(backup, out_path); print('khôi phục từ Drive:', backup)
    done = {r['image_id'] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in rows if r['image_id'] not in done]
    print(f'{out_path}: đã có {len(done)}, còn {len(todo)} ảnh')
    t0 = time.time()
    with open(out_path, 'a', encoding='utf-8') as f:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            images = [Image.open(DATA / r['image_path']).convert('RGB') for r in batch]
            for r, raw in zip(batch, generate_batch(images, prompt, max_new_tokens)):
                f.write(json.dumps({'image_id': r['image_id'], 'raw': raw}, ensure_ascii=False) + '\n')
            f.flush()
            n = i + len(batch)
            if (i // batch_size) % 10 == 9 or n == len(todo):
                el = time.time() - t0
                print(f'  {n}/{len(todo)} ảnh | {el / 60:.1f} phút | còn ≈ {el / n * (len(todo) - n) / 60:.1f} phút')
                if DRIVE_DIR.parent.exists():
                    DRIVE_DIR.mkdir(parents=True, exist_ok=True); shutil.copy(out_path, backup)
    return (time.time() - t0) / 60

# Thử trước trên 2 ảnh val: số token ảnh (640×360 phải là 220) và output thô
val = read_jsonl(DATA / 'splits' / 'val.jsonl')
im = Image.open(DATA / val[0]['image_path']).convert('RGB')
enc = processor.apply_chat_template([{'role': 'user', 'content': [{'type': 'image', 'image': im},
                                     {'type': 'text', 'text': PROMPT}]}],
                                    tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
img_pad = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
print('\nsố token ảnh:', int((enc['input_ids'] == img_pad).sum()), '| tổng token prompt:', enc['input_ids'].shape[1])
for r, raw in zip(val[:2], generate_batch([Image.open(DATA / r['image_path']).convert('RGB') for r in val[:2]], PROMPT)):
    print(f"\n[{r['image_id']}] GT : {objects_to_answer(r['objects'])}\n  raw: {raw!r}")

# Chạy E0 trên toàn bộ test
test = read_jsonl(DATA / 'splits' / 'test.jsonl')
minutes = predict(test, 'work/e0.jsonl', PROMPT)
print(f'\nE0 xong: {minutes:.1f} phút | VRAM đỉnh {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

In [ ]:
# 3b. Phân tích lỗi E0 (Câu 2): loại lỗi, thang toạ độ, 5 output thô
from collections import Counter
from scripts.score import parse_output, evaluate
from scripts.labcommon import CLASSES, read_jsonl

test = read_jsonl('data/splits/test.jsonl')
preds = {r['image_id']: r['raw'] for r in read_jsonl('work/e0.jsonl')}
ev = json.load(open('report/evidence/e0.json', encoding='utf-8'))
print(f"parse_valid_rate = {ev['parse_valid_rate']:.3f} | F1 = {ev['micro']['f1']:.3f} "
      f"CI95 = {ev['f1_ci95']} | FP/ảnh = {ev['fp_per_image']:.2f} | mean IoU = {ev['mean_iou']:.3f}")

kinds, labels, n_pred, n_gt, coords = Counter(), Counter(), 0, 0, []
for e in test:
    raw = preds[e['image_id']]
    p = parse_output(raw)
    n_gt += len(e['objects'])
    if '```' in raw: kinds['bọc ```json (score.py vẫn đọc được)'] += 1
    if p is None:
        kinds['JSON hỏng'] += 1
        if raw.count('[') > raw.count(']') or raw.count('{') > raw.count('}'):
            kinds['  ↳ bị cắt giữa chừng (chưa đóng ] hoặc })'] += 1
        continue
    n_pred += len(p)
    labels.update(q['label'] for q in p)
    coords += [q['bbox'] for q in p]
    if not p and e['objects']: kinds['trả [] dù ảnh có hư hại'] += 1
    if any(q['label'] not in CLASSES for q in p): kinds['có nhãn ngoài 3 lớp'] += 1
    if len(p) > len(e['objects']): kinds['nhiều box hơn GT'] += 1
    if len(p) < len(e['objects']): kinds['ít box hơn GT'] += 1
    if len({q['bbox'] for q in p}) < len(p): kinds['lặp lại cùng một box'] += 1
print(f'\n{len(test)} ảnh test | box dự đoán {n_pred} vs box thật {n_gt}')
for k, v in kinds.most_common():
    print(f'  {k:45s} {v:4d} ảnh')
print('nhãn dự đoán:', dict(labels.most_common()))

# Thang toạ độ: nếu model trả pixel (x ≤ 640, y ≤ 360) thì chấm --coord pixel sẽ cho F1 cao hơn
if coords:
    xs = [v for b in coords for v in (b[0], b[2])]; ys = [v for b in coords for v in (b[1], b[3])]
    print(f'toạ độ dự đoán: x ∈ [{min(xs):.0f}, {max(xs):.0f}], y ∈ [{min(ys):.0f}, {max(ys):.0f}]')
for coord in ('norm1000', 'pixel'):
    m = evaluate(test, preds, coord=coord, n_boot=200)
    print(f"  chấm E0 với --coord {coord:8s}: F1 = {m['micro']['f1']:.3f}  mean IoU = {m['mean_iou']:.3f}")

print('\n5 output thô đầu tiên:')
for e in test[:5]:
    print(f"\n[{e['image_id']}] GT : {objects_to_answer(e['objects'])}\n  raw: {preds[e['image_id']][:400]!r}")

# Sao lưu bằng chứng sang Drive (và tải về máy để đưa vào repo)
if DRIVE_DIR.parent.exists():
    shutil.copy('report/evidence/e0.json', DRIVE_DIR / 'e0.json'); print('\nđã sao lưu e0.json sang', DRIVE_DIR)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9